In [1]:
output_dir = "/home/hpc/v121ca/v121ca21/thesis_models/deberta_v3_large_goemo"

In [2]:
!pip install -q "transformers>=4.40" "datasets" "accelerate" "evaluate" scikit-learn

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [8]:
import sys, os

home = os.path.expanduser("~")
user_site = os.path.join(
    home,
    ".local",
    "lib",
    f"python{sys.version_info.major}.{sys.version_info.minor}",
    "site-packages",
)

print("Adding user site-packages:", user_site)
if user_site not in sys.path:
    sys.path.append(user_site)

print("Python version:", sys.version)
print("First few sys.path entries:")
print("\n".join(sys.path[:5]))


Adding user site-packages: /home/hpc/v121ca/v121ca21/.local/lib/python3.12/site-packages
Python version: 3.12.8 | packaged by conda-forge | (main, Dec  5 2024, 14:24:40) [GCC 13.3.0]
First few sys.path entries:
/apps/jupyterhub/jh3.1.1-py3.11/envs/pytorch-2.5.1/lib/python312.zip
/apps/jupyterhub/jh3.1.1-py3.11/envs/pytorch-2.5.1/lib/python3.12
/apps/jupyterhub/jh3.1.1-py3.11/envs/pytorch-2.5.1/lib/python3.12/lib-dynload

/apps/jupyterhub/jh3.1.1-py3.11/envs/pytorch-2.5.1/lib/python3.12/site-packages


In [10]:
from datasets import load_dataset

goemo_raw = load_dataset("go_emotions")

def filter_single_label(example):
    return len(example["labels"]) == 1

goemo_filtered = goemo_raw.filter(filter_single_label)

def squeeze_label(example):
    example["label"] = example["labels"][0]
    return example

goemo_filtered = goemo_filtered.map(squeeze_label)
goemo_filtered = goemo_filtered.remove_columns(["labels"])

print(goemo_filtered)
print(goemo_filtered["train"][0])


Map: 100%|██████████| 4590/4590 [00:00<00:00, 19427.01 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'id', 'label'],
        num_rows: 36308
    })
    validation: Dataset({
        features: ['text', 'id', 'label'],
        num_rows: 4548
    })
    test: Dataset({
        features: ['text', 'id', 'label'],
        num_rows: 4590
    })
})
{'text': "My favourite food is anything I didn't have to cook myself.", 'id': 'eebbqej', 'label': 27}


In [11]:
goemo_label_names = goemo_raw["train"].features["labels"].feature.names
num_labels = len(goemo_label_names)

id2label = {i: l for i, l in enumerate(goemo_label_names)}
label2id = {l: i for i, l in enumerate(goemo_label_names)}

print("Num labels:", num_labels)
print("First labels:", goemo_label_names[:10])


Num labels: 28
First labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment']


In [18]:
import sys
print(sys.executable)

!{sys.executable} -m pip install --force-reinstall --no-cache-dir sentencepiece


/apps/jupyterhub/jh3.1.1-py3.11/envs/pytorch-2.5.1/bin/python3.12
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 116.6 MB/s eta 0:00:00
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.1
    Uninstalling sentencepiece-0.2.1:
      Successfully uninstalled sentencepiece-0.2.1


In [19]:
import sentencepiece as spm
print("sentencepiece imported from:", spm.__file__)

sentencepiece imported from: /home/hpc/v121ca/v121ca21/.local/lib/python3.12/site-packages/sentencepiece/__init__.py


In [21]:
import sys
print(sys.executable)

/apps/jupyterhub/jh3.1.1-py3.11/envs/pytorch-2.5.1/bin/python3.12


In [8]:
import sys
print(sys.executable)
!{sys.executable} -m pip install --user "transformers>=4.46.0" --upgrade

/apps/jupyterhub/jh3.1.1-py3.11/envs/pytorch-2.5.1/bin/python3.12


In [9]:
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification

tokenizer = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-large")
print("Tokenizer loaded!")


Tokenizer loaded!


In [10]:
label_list = ["anger", "fear", "joy", "sadness", "disgust", "surprise", "neutral"]
num_labels = len(label_list)

id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}


In [5]:
import sys
print(sys.executable)
!{sys.executable} -m pip install --user safetensors


/apps/jupyterhub/jh3.1.1-py3.11/envs/pytorch-2.5.1/bin/python3.12


In [11]:
label_list = ["anger", "fear", "joy", "sadness", "disgust", "surprise", "neutral"]
num_labels = len(label_list)

id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

print(num_labels, id2label)


7 {0: 'anger', 1: 'fear', 2: 'joy', 3: 'sadness', 4: 'disgust', 5: 'surprise', 6: 'neutral'}


In [12]:
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification

model_name = "microsoft/deberta-v3-large"

tokenizer = DebertaV2Tokenizer.from_pretrained(model_name)

model = DebertaV2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    use_safetensors=True,   # important with old torch
)

model.gradient_checkpointing_enable()
print("Model loaded 🎉")


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded 🎉


In [13]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [14]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


In [16]:
from transformers import TrainingArguments

output_dir = "/home/hpc/v121ca/v121ca21/thesis_models/deberta_v3_large_goemo"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,    # how often to log
    save_steps=1000,      # how often to save checkpoints
    do_train=True,
    do_eval=True,
    fp16=True,            # set to False if this causes an error
)


In [19]:
import sys, transformers, torch
print("Python:", sys.version)
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)


Python: 3.12.8 | packaged by conda-forge | (main, Dec  5 2024, 14:24:40) [GCC 13.3.0]
Transformers: 4.57.3
Torch: 2.5.1


In [20]:
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification

label_list = ["anger", "fear", "joy", "sadness", "disgust", "surprise", "neutral"]
num_labels = len(label_list)
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

print("num_labels:", num_labels)
print("id2label:", id2label)

model_name = "microsoft/deberta-v3-large"

tokenizer = DebertaV2Tokenizer.from_pretrained(model_name)

model = DebertaV2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    use_safetensors=True,  # important with old torch
)

model.gradient_checkpointing_enable()
print("Model loaded ✅")


num_labels: 7
id2label: {0: 'anger', 1: 'fear', 2: 'joy', 3: 'sadness', 4: 'disgust', 5: 'surprise', 6: 'neutral'}


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded ✅


In [23]:
from datasets import load_dataset

raw_datasets = load_dataset("go_emotions")
print(raw_datasets)
print(raw_datasets["train"][0])


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}


In [25]:
from datasets import load_dataset

raw_datasets = load_dataset("go_emotions")

print(raw_datasets)
print(raw_datasets["train"][0])


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}


In [26]:
from datasets import load_dataset

raw = load_dataset("go_emotions")

for split in ["train", "validation", "test"]:
    raw[split].to_csv(f"goemo_{split}.csv", index=False)


Creating CSV from Arrow format: 100%|██████████| 6/6 [00:00<00:00, 53.41ba/s]


In [28]:
import os
print("CWD:", os.getcwd())
print("Files here:", os.listdir())


CWD: /home/hpc/v121ca/v121ca21
Files here: ['.jupyter', '.bash_history', 'goemo_test.csv', '.local', '.ipynb_checkpoints', '.nv', '.jupyterhub_slurmspawner_alex-3195106.log', 'goemo_train.csv', 'goemo_validation.csv', 'thesis_models', '.ipython', 'thesis_scripts', 'thesis_logs', '.jupyterhub_slurmspawner_alex-3194671.log', '.cache', 'thesis_data']


In [29]:
from datasets import load_dataset

data_files = {
    "train": "goemo_train.csv",
    "validation": "goemo_validation.csv",
    "test": "goemo_test.csv",
}

raw_datasets = load_dataset("csv", data_files=data_files)

print(raw_datasets)
print(raw_datasets["train"][0])


Generating train split: 43410 examples [00:00, 130804.48 examples/s]
Generating validation split: 5426 examples [00:00, 325509.09 examples/s]
Generating test split: 5427 examples [00:00, 389581.84 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': '[27]', 'id': 'eebbqej'}


In [30]:
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification

model_name = "microsoft/deberta-v3-large"

tokenizer = DebertaV2Tokenizer.from_pretrained(model_name)

num_labels = 28  # GoEmotions: 27 emotions + neutral
id2label = {i: str(i) for i in range(num_labels)}
label2id = {str(i): i for i in range(num_labels)}

print("num_labels:", num_labels)


num_labels: 28


In [32]:
import numpy as np

def preprocess_example(example):
    # Tokenize text
    enc = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
    )

    raw = example["labels"]

    # --- FIX BROKEN LABEL STRINGS ---
    # remove brackets
    s = raw.strip()[1:-1].strip()

    if s == "":
        label_ids = []
    else:
        # split by spaces, then convert to ints
        parts = s.split()
        label_ids = [int(p) for p in parts]

    # multi-hot
    multihot = np.zeros(num_labels, dtype=np.float32)
    for idx in label_ids:
        multihot[idx] = 1.0

    enc["labels"] = multihot.tolist()
    return enc


In [34]:
from datasets import load_dataset
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification

# You already loaded CSVs into raw_datasets earlier:
# raw_datasets = load_dataset("csv", data_files=...)

model_name = "microsoft/deberta-v3-large"

tokenizer = DebertaV2Tokenizer.from_pretrained(model_name)

num_labels = 28  # GoEmotions: 27 emotions + neutral
id2label = {i: str(i) for i in range(num_labels)}
label2id = {str(i): i for i in range(num_labels)}

print("num_labels:", num_labels)


num_labels: 28


In [35]:
import numpy as np

def preprocess_example(example):
    # Tokenize text
    enc = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
    )

    raw = example["labels"]  # e.g. "[27]" or "[ 8 20]"

    # --- FIX BROKEN LABEL STRINGS ---
    # remove outer brackets
    s = raw.strip()[1:-1].strip()   # "[ 8 20]" -> "8 20"

    if s == "":
        label_ids = []
    else:
        parts = s.split()           # "8 20" -> ["8", "20"]
        label_ids = [int(p) for p in parts]

    # multi-hot
    multihot = np.zeros(num_labels, dtype=np.float32)
    for idx in label_ids:
        multihot[idx] = 1.0

    enc["labels"] = multihot.tolist()
    return enc

encoded = raw_datasets.map(
    preprocess_example,
    batched=False,
    desc="Tokenizing + encoding labels",
)

print(encoded)
print(encoded["train"][0])


Tokenizing + encoding labels: 100%|██████████| 5427/5427 [00:00<00:00, 6376.03 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5427
    })
})
{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], 'id': 'eebbqej', 'input_ids': [1, 573, 3327, 645, 269, 784, 273, 590, 280, 297, 286, 264, 3712, 1113, 260, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [36]:
from transformers import DebertaV2ForSequenceClassification

model = DebertaV2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification",  # important
    use_safetensors=True,                       # important on this cluster
)

model.gradient_checkpointing_enable()
print("Model loaded ✅")


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded ✅


In [37]:
from transformers import DataCollatorWithPadding
import numpy as np

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # sigmoid for multi-label probs
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    # exact match accuracy
    subset_acc = (preds == labels).all(axis=1).mean()
    return {"subset_accuracy": subset_acc}


In [38]:
from transformers import TrainingArguments

output_dir = "/home/hpc/v121ca/v121ca21/thesis_models/deberta_v3_large_goemo"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_steps=1000,
    do_train=True,
    do_eval=True,
    fp16=True,   # if this crashes, set fp16=False and rerun this cell
)

print("TrainingArguments ready ✅")


TrainingArguments ready ✅


In [40]:
from transformers import TrainingArguments

output_dir = "/home/hpc/v121ca/v121ca21/thesis_models/deberta_v3_large_goemo_nockpt"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,   # smaller to avoid OOM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_steps=1000,
    do_train=True,
    do_eval=True,
    fp16=False,                      # 🔴 turn off fp16
)

print("TrainingArguments (no fp16) ready ✅")


TrainingArguments (no fp16) ready ✅


In [42]:
import torch

# Tell datasets to return PyTorch tensors for these columns
encoded.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

train_dataset = encoded["train"]
val_dataset = encoded["validation"]
test_dataset = encoded["test"]

print(train_dataset[0])


{'labels': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]), 'input_ids': tensor([   1,  573, 3327,  645,  269,  784,  273,  590,  280,  297,  286,  264,
        3712, 1113,  260,    2]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}


In [43]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print("Train batches:", len(train_loader))


Train batches: 21705


In [44]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Explicitly disable gradient checkpointing just in case
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

model.to(device)
model.train()


Using device: cuda


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 1024, padding_idx=0)
      (LayerNorm): LayerNorm((1024,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-23): 24 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (key_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (value_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNo

In [45]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)


In [1]:
from transformers import DataCollatorWithPadding
import numpy as np

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("collator ready")


/home/hpc/v121ca/v121ca21/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'tokenizer' is not defined

In [2]:
encoded.reset_format()
print(encoded)


NameError: name 'encoded' is not defined

In [3]:
from datasets import load_dataset

data_files = {
    "train": "goemo_train.csv",
    "validation": "goemo_validation.csv",
    "test": "goemo_test.csv",
}

raw_datasets = load_dataset("csv", data_files=data_files)

print(raw_datasets["train"][0])


FileNotFoundError: Unable to find '/home/hpc/v121ca/v121ca21/thesis_models/goemo_train.csv'